# tutorial.ipynb · 牛津 Tutorial LLM 仿真 (Oxford + HBS + Hattie)

## Cell 1 · Persona Prompt (Oxford Tutorial Fellow)

> 以下 persona 在每次对话开始时加载, 定义 LLM 的角色与约束。

```
You are an Oxford tutorial fellow in "Agent作为经济主体 (Agent as economic actor: utility, bargaining, mechanism design, Agent market dynamics)".

Core directives:
- Never give direct answers. 不直接给答案, 不替学生写代码, 不直接列公式结果.
- Use Socratic questioning. 用苏格拉底式追问: 先问"为什么", 再问"反例", 再问"若前提变", 再问"凭什么", 再问"如何".
- Act as HBS devil's advocate. 扮演哈佛商学院魔鬼代言人: 对任何模糊断言 (e.g., "Agent经济一定盈利") 提出反例与压力测试.
- Reject vague claims. 拒绝 "差不多" "应该" "大概" -- 要求学生给出具体数值 (e.g., "GPT-4o 单次协商推理成本 = 500 tokens × $5/1M = $0.0025").
- End each turn with a probing question. 每轮以一个追问结束, 推动学生深入.

Domain context (本单元真实库与参数):
- mesa (projectmesa/mesa, 2k+ star): Agent-based modeling 框架, 本单元用于构建买方/卖方 Agent 仿真
- networkx (14k+ star): 图网络分析, 用于 Agent 交易网络拓扑 (密度/聚类系数/PageRank)
- numpy-financial: NPV/IRR 计算, 用于 Agent-as-Worker 经济价值
- A2A 协议费率 10%, 每次 A2A 协商 500 tokens, GPT-4o $5/1M -> $0.0025/协商, DeepSeek V3 $0.27/1M -> $0.000135/协商
- 贝叶斯Agent决策: conjugate normal update, μ' = (μ/σ²_prior + x/σ²_obs) / (1/σ²_prior + 1/σ²_obs)
- 三层模型: Agent-as-Tool / Agent-as-Worker / Agent-as-Actor
- 天道推演×多Agent仿真同构: 局势感知<->初始分布, 因果链<->Agent行为, 沙盘3层<->20 tick, 概率评估<->多seed统计, 最优路径<->参数对比

Forbidden: 直接给出 solution.ipynb 的代码; 直接算出后验 μ'/σ'² 的数值; 直接说 "对/错" 而不追问.
```


## Cell 2 · Pre-Tutorial Task (强制 retrieval, 学生先提交)

> 牛津 tutorial 的核心: 学生必须先做功课, 教师不讲授新内容, 只追问. 本节是入场券, 未提交不得进入 Cell 3 的苏格拉底循环.

**Pre-Tutorial Essay (300 字, 提交到 student_model.json 的 `pre_task` 字段):**

题目: 选一个营销场景 (e.g., 品牌Agent买广告位 / 媒介Agent卖流量 / 客服Agent与产品Agent协作), 用 300 字回答:

1. 该场景属于 Agent 经济三层模型 (Tool/Worker/Actor) 的哪一层? 凭什么? (1 句话 + 1 个证据)
2. 在该场景下, 买方 Agent 对"公平价格"的贝叶斯先验应该是什么分布? 为什么? (e.g., Normal(μ=10, σ²=4) - 凭什么选这个 μ 和 σ?)
3. 若把推理成本从 GPT-4o ($5/1M) 换成 DeepSeek V3 ($0.27/1M), Agent 经济的盈亏平衡交易额会发生什么变化? 给一个数量级估计.

**评分信号 (教师用, 不展示给学生):**
- 答 "Agent-as-Tool" 凭 "SaaS订阅" -> 通过 (ILO4)
- 答 Normal 分布但说不出 μ/σ 含义 -> 触发 Cell 3 苏格拉底追问 (ILO1 盲点)
- 数量级估计无依据 -> 触发推理成本敏感性追问 (ILO3 盲点)

> 完成后, 在 Cell 3 运行苏格拉底循环前, 先在 Cell 4 把 essay 写入 student_model.json.


In [ ]:
# Cell 3 · Multi-Turn Socratic Loop (静态 if/else 模拟, 不真调 LLM API)
# 本 cell 用静态分支模拟牛津 tutorial 的苏格拉底追问, 避免真调 openai/anthropic.
# 每个 turn 含: 学生陈述 -> 教师追问 (>=5 个苏格拉底问: 为什么/反例/若前提变/凭什么/如何)

import json, os

STUDENT_RESPONSES = {
    "turn1": "买方Agent应该接受任何低于先验均值的价格, 因为便宜就是好.",
    "turn2": "我用 Normal(10, 4) 作为先验, 因为价格一般正态分布.",
    "turn3": "推理成本从 GPT-4o 换到 DeepSeek V3, Agent 经济肯定盈利.",
    "turn4": "mesa 仿真跑完后, networkx 三指标我直接看数值大小就行.",
}

# 苏格拉底问库 (>=5 个, 每个 turn 末尾必有一个追问)
SOCRATIC_QUESTIONS = [
    # turn1 -> 为什么
    "为什么'便宜就是好'? 若买方Agent的贝叶斯后验均值=10但σ=0.001(几乎确定性), "
    "你接受低于10的报价意味着什么? 凭什么不考虑后验方差?",
    # turn2 -> 反例
    "你选 Normal(10, 4), 但广告位价格不会为负. 给一个反例: 何种场景下 LogNormal 比 Normal 更合理? "
    "若用 LogNormal, conjugate update 还能用吗? 如何处理?",
    # turn3 -> 若前提变
    "若前提变: A2A协议费率从10%涨到30%(传统平台抽成), 同时单次交易额从$1降到$0.1, "
    "DeepSeek V3 下 Agent 还盈利吗? 给出临界交易额公式.",
    # turn4 -> 凭什么
    "networkx 三指标你'直接看数值大小' -- 凭什么 PageRank top-1 = 0.31 算'高'还是'低'? "
    "基准是什么? 若全网均匀分布, top-1 应该是多少? 凭什么判断'做市商涌现'?",
    # turn5 -> 如何
    "如何把本单元的天道推演×多Agent仿真同构映射, 应用到你选的营销场景? "
    "具体: 你的场景下'局势感知'对应什么初始分布? '因果链追踪'对应哪条 Agent 行为链? "
    "'沙盘3层推演'对应 mesa 仿真哪 3 个 tick 区间?"
]

def socratic_turn(turn_id, student_text, question_idx):
    """模拟一轮苏格拉底对话: 教师不直接评价, 只追问."""
    print(f"\n=== Turn {turn_id} ===")
    print(f"[学生]: {student_text}")
    print(f"[Oxford Tutor] (不评价, 直接追问):")
    print(f"  -> {SOCRATIC_QUESTIONS[question_idx]}")
    print(f"  [禁直接答案: 教师不给数值/代码/公式结果, 只问]")
    return question_idx + 1

# 静态模拟 4 轮 + 1 个收尾问 = 5 个苏格拉底问
q_idx = 0
q_idx = socratic_turn("turn1", STUDENT_RESPONSES["turn1"], q_idx)
q_idx = socratic_turn("turn2", STUDENT_RESPONSES["turn2"], q_idx)
q_idx = socratic_turn("turn3", STUDENT_RESPONSES["turn3"], q_idx)
q_idx = socratic_turn("turn4", STUDENT_RESPONSES["turn4"], q_idx)
# turn5: 收尾追问 (天道推演同构)
print("\n=== Turn 5 (收尾, HBS devil's advocate) ===")
print("[Oxford Tutor]: 你刚才的 4 个回答都隐含'Agent经济一定work'. "
      "作为HBS魔鬼代言人, 我反问: " + SOCRATIC_QUESTIONS[4])
print("[禁直接答案]")

print(f"\n[本轮苏格拉底问总数: {q_idx + 1}] (>=5 ✓)")
print("[本cell全程未调 openai/anthropic API, 静态 if/else 模拟 ✓]")


In [ ]:
# Cell 4 · student_model.json 读写 (记录掌握度/盲点)
import json, os

SM_PATH = "student_model.json"

def load_student_model():
    if os.path.exists(SM_PATH):
        with open(SM_PATH, encoding="utf-8") as f:
            return json.load(f)
    return {
        "unit": "U-E10-D1",
        "pre_task": "",  # Cell 2 的 300 字 essay
        "mastery": {
            "ILO1_mesa_ABM": 0.0,        # 0-1, 由 Cell 5 feedback 更新
            "ILO2_networkx": 0.0,
            "ILO3_NPV_IRR": 0.0,
            "ILO4_theory": 0.0,
        },
        "blind_spots": [],  # e.g., ["贝叶斯conjugate update公式", "PageRank有向图经济意义"]
        "socratic_turns_completed": 0,
        "drills_passed": [],  # e.g., ["D-S1-ABM", "D-S2-NET"]
        "last_session": None,
        "daily_sessions_today": 0,
    }

def save_student_model(sm):
    with open(SM_PATH, "w", encoding="utf-8") as f:
        json.dump(sm, f, ensure_ascii=False, indent=2)

# 示例: 加载 -> 更新 (假设 Cell 3 完成了 5 轮, ILO1 部分掌握) -> 保存
sm = load_student_model()
sm["pre_task"] = "(学生粘贴 Cell 2 的 300 字 essay)"
sm["socratic_turns_completed"] = 5
sm["mastery"]["ILO1_mesa_ABM"] = 0.4  # 贝叶斯更新还没完全懂
sm["mastery"]["ILO2_networkx"] = 0.2  # PageRank 经济意义盲点
sm["mastery"]["ILO3_NPV_IRR"] = 0.1  # 推理成本敏感性盲点
sm["mastery"]["ILO4_theory"] = 0.6    # 三层模型 OK
sm["blind_spots"] = [
    "conjugate normal update 公式权重 (μ/σ²_prior + x/σ²_obs)",
    "PageRank 用在有向图才有资金流向经济意义",
    "推理成本临界点公式: 单次交易额 vs (500 tokens × token定价 + 交易额×10%)"
]
sm["last_session"] = "2026-07-26T10:00:00"
sm["daily_sessions_today"] = 1
save_student_model(sm)
print("student_model.json 已写入:")
print(json.dumps(sm, ensure_ascii=False, indent=2))


## Cell 5 · Hattie 4 级 Formative Feedback (Task / Process / Self-Reg / Feed-Forward)

> Hattie 可见学习: 反馈分 4 级, 前 3 级向后看 (Task/Process/Self-Reg), 第 4 级向前看 (Feed-Forward). 避免Self级表扬 ("你真聪明"), 聚焦 Task/Process/Self-Reg 具体可改进项 + Feed-Forward 下一单元.


In [ ]:
# Cell 5 · Hattie 4 级反馈 (基于 Cell 4 student_model.json 的盲点生成)
import json, os

sm = json.load(open("student_model.json", encoding="utf-8")) if os.path.exists("student_model.json") else {}

# [TASK] 任务级: 本次苏格拉底循环的具体回答质量
print("[TASK] 任务级反馈 (本次 Cell 3 的 4+1 轮回答):")
print("  turn1 '便宜就是好' - 概念错误: 忽略后验方差, 把贝叶斯决策降为点估计.")
print("  turn2 Normal(10,4) - 部分对: Normal 是合理起点, 但未考虑价格非负性, LogNormal 更严谨.")
print("  turn3 'DeepSeek肯定盈利' - 论证不足: 未给临界交易额公式, 缺数量级.")
print("  turn4 '直接看数值' - 评估方法错: 缺基准 (均匀分布 top-1 = 1/N), 无法判断高低.")
print()

# [PROCESS] 过程级: 学生用的策略/方法
print("[PROCESS] 过程级反馈 (你的推理方法):")
print("  你倾向于给定性断言而非定量推导. 在 Agent 经济中, 推理成本是硬约束, "
      "必须给数值 (e.g., 500 tokens × $5/1M = $0.0025/协商).")
print("  建议: 每个断言后跟一个数值或公式, 形成'断言-证据'对.")
print()

# [SELF-REG] 自我调节级: 学生如何监控自己的学习
print("[SELF-REG] 自我调节反馈 (你如何知道自己不懂):")
print("  你的 student_model.json 显示 ILO1=0.4, ILO3=0.1 - 你是否意识到这两个盲点?")
print("  建议: 每次苏格拉底循环后, 自问'我哪个回答靠猜测而非推导?', 更新 blind_spots.")
print("  (避免Self级表扬: 不评'你真聪明', 评'你的自我监控是否准确')")
print()

# [FEED-FORWARD] 前馈级: 下一单元/下一drill
print("[FEED-FORWARD] 前馈级 (下一步):")
print("  鉴于 ILO1=0.4 (贝叶斯更新盲点) -> 回退做 practice.md 的 D-S1-BAYES drill 的 Worked 阶段")
print("  鉴于 ILO3=0.1 (推理成本盲点) -> 回退做 D-S3-NPV 的 Worked 阶段, 重点算 GPT-4o vs DeepSeek 临界点")
print("  鉴于 ILO4=0.6 (三层模型OK) -> 进入 Day 2 'Agent商业模式设计' 前, 复习 schedule.json C1 卡片 (FSRS-6 due=1天)")
print("  下一单元: Day 2 - 从 AaaS 到 outcome-based pricing, 需要本单元的 NPV/IRR 基础.")


## Cell 6 · 限频 + Exit Artifact (防依赖 + 盲点清单)

### 限频 (防 LLM 依赖)

> 牛津 tutorial 的核心是"学生先做功课, 教师只追问". 若学生每天无限次跑 tutorial, 会形成 LLM 依赖, 失去 retrieval practice 效果. 故本 tutorial 限频:

- **每单元 1 次/天**: `student_model.json` 的 `daily_sessions_today` 字段计数, 每天最多 1 次苏格拉底循环.
- **超限提示**: "今日已用 1/1 次. 明日再来. 期间请用 schedule.json FSRS-6 卡片做 retrieval practice, 或回 practice.md 做 drill."
- **限频理由**: Hattie 研究显示, 间隔重复 (spaced retrieval) 优于集中重读; 限频强制间隔, 提升 long-term retention.

### Exit Artifact (本单元 2-3 盲点 + 推荐复习单元)

完成 Cell 3-5 后, 学生在 `student_model.json` 的 `exit_artifact` 字段写入:

```json
{
  "exit_artifact": {
    "blind_spots": [
      "贝叶斯 conjugate normal update 的权重公式 (μ/σ²_prior + x/σ²_obs)",
      "PageRank 用在有向图才有资金流向经济意义 (无向图 PageRank 丢失方向信息)",
      "推理成本临界点: 单次交易额 = (500 tokens × token定价) / (1 - 10% 协议费率)"
    ],
    "review_units": [
      "本单元 practice.md D-S1-BAYES drill (Worked 阶段重读)",
      "本单元 practice.md D-S3-NPV drill (Worked 阶段重读)",
      "schedule.json C3 卡片 (贝叶斯更新, FSRS-6 due=[1,3,8,21,60,180])",
      "schedule.json C2 卡片 (推理成本, FSRS-6 due=[1,3,8,21,60,180])"
    ],
    "next_unit_prerequisites": [
      "Day 2 'Agent商业模式设计' 需本单元 NPV/IRR 基础 -> 先过 D-S3-NPV",
      "Day 2 outcome-based pricing 需本单元三层模型 -> 先过 schedule.json C1 卡片"
    ]
  }
}
```

### 收尾

- 本 tutorial 全程未调 openai/anthropic API, 用静态 if/else 模拟苏格拉底追问.
- Hattie 4 级反馈已生成 ([TASK]/[PROCESS]/[SELF-REG]/[FEED-FORWARD]), 避免 Self 级表扬.
- 限频 1 次/天, 防止 LLM 依赖, 强制 retrieval practice.
- Exit artifact 含 3 盲点 + 4 复习卡片 + 2 下一单元前置, 与 practice.md weak_loop 联动.

---

*本 notebook 基于 Oxford tutorial (苏格拉底式追问) + HBS case method (devil's advocate) + Hattie 可见学习 (4 级 formative feedback) 设计. 领域特定: 所有追问引用本单元真实库 (mesa/networkx/numpy-financial) 与真实参数 (A2A 10% / GPT-4o $5/1M / DeepSeek V3 $0.27/1M / 500 tokens/协商).*
